In [1]:
import pandas as pd 

In [2]:
DSF = pd.read_csv('../data/DietarySupplementFacts_8.csv')
PO = pd.read_csv('../data/ProductOverview_8.csv')

In [25]:
DSF = DSF[['DSLD ID','Ingredient']]
DSF.head()

,DSLD ID,Ingredient
0,332294,Calories
1,332294,Total Carbohydrates
2,332294,Total Sugars
3,332294,Added Sugars
4,332294,Melatonin


In [58]:
DSF_grouped = DSF.groupby('DSLD ID')['Ingredient'].apply(', '.join).reset_index()

In [39]:
DSF.shape

(44416, 2)

In [59]:
DSF_grouped

,DSLD ID,Ingredient
0,332294,"Calories, Total Carbohydrates, Total Sugars, A..."
1,332295,"Calories, Total Carbohydrates, Total Sugars, A..."
2,332296,"Calories, Total Carbohydrates, Total Sugars, A..."
3,332297,"Calories, Total Carbohydrates, Total Sugars, A..."
4,332298,"Calories, Total Carbohydrates, Total Sugars, A..."
...,...,...
4775,341410,"Calories, Total Carbohydrate, Total Sugars, Ad..."
4776,341428,"Vitamin C, Vitamin B3, Vitamin B6, Folate, Fol..."
4777,341498,"Whey Protein Isolate, Milk Protein Isolate, Cr..."
4778,341509,"Vitamin A, Vitamin C, Calcium, Magnesium, Sodi..."


In [40]:
DSF_grouped.shape

(4780, 2)

In [56]:
PO = PO[['DSLD ID','Brand Name','Suggested Use']]
PO.head()

,DSLD ID,Brand Name,Suggested Use
0,332294,Natrol,Suggested use: Adults; take 1 gummy 30 minutes...
1,332295,Natrol,Suggested use: Adults; take 1 gummy 30 minutes...
2,332296,Natrol,Suggested use: Adults; take 1 gummy 30 minutes...
3,332297,Natrol,Suggested Use: Adults; take 1 gummy 30 minutes...
4,332298,Natrol,Suggested Use: Adults; take 1 gummy 30 minute...


In [12]:
PO.shape

(4780, 3)

In [5]:
file_path = "../data/LabelStatements_8.csv"

rows = []
current = None

with open(file_path, "r", encoding="utf-8", errors="replace") as f:
    header = next(f)  # skip header

    for line in f:
        line = line.rstrip("\n")

        if line.startswith("https://"):
            # save previous record
            if current is not None:
                rows.append(current)

            # split only first 4 commas -> 5 fields total
            parts = line.split(",", 4)

            if len(parts) == 5:
                current = parts
            else:
                current = None
        else:
            # continuation of Statement field
            if current is not None:
                current[4] += "\n" + line

# save last record
if current is not None:
    rows.append(current)

LS = pd.DataFrame(rows, columns=[
    "URL", "DSLD ID", "Product Name", "Statement Type", "Statement"
])

In [8]:
LS = LS[['DSLD ID','Product Name','Statement']]

In [9]:
LS.head()

,DSLD ID,Product Name,Statement
0,332294,Melatonin 5 mg Strawberry,Great value trusted brand\nSleep #1 drug-free ...
1,332294,Melatonin 5 mg Strawberry,Fall asleep faster stay asleep longer\nWake up...
2,332294,Melatonin 5 mg Strawberry,Dietary Supplement These statements have not b...
3,332294,Melatonin 5 mg Strawberry,Suggested use: Adults take 1 gummy 30 minutes ...
4,332294,Melatonin 5 mg Strawberry,Warning: Not intended for individuals under th...


In [13]:
LS.shape

(30743, 3)

In [22]:
LS_grouped = LS.groupby('DSLD ID' , as_index=False).agg({'Product Name': 'first', 'Statement' : ', '.join})

In [55]:
LS_grouped

,DSLD ID,Product Name,Statement
0,332294,Melatonin 5 mg Strawberry,Great value trusted brand\nSleep #1 drug-free ...
1,332295,Melatonin 5 mg Strawberry,Great value trusted brand\nSleep #1 drug-free ...
2,332296,Melatonin 5 mg Strawberry,Great value trusted brand\nSleep #1 drug-free ...
3,332297,Melatonin 5 mg Strawberry,Sleep\nFall asleep faster\nStay asleep longer\...
4,332298,MelatoninMax 10 mg Blueberry,Sleep\nMaximum strength\nGet better sleep\nWak...
...,...,...,...
4775,341410,CranRx Liquid Cranberry,Fast absorbing liquid Bioactive cranberry Extr...
4776,341428,Blood Balance,Suggested use: As a dietary supplement take on...
4777,341498,Muscle 5 french Vanilla Flavour,24 g Whey protein\n16 g Milk protein\n3 g Crea...
4778,341509,Congaplex,"Immune health, Dietary Supplement This stateme..."


In [60]:
LS_grouped['DSLD ID'] = LS_grouped['DSLD ID'].astype(str)
PO['DSLD ID'] = PO['DSLD ID'].astype(str)
DSF_grouped['DSLD ID'] = DSF_grouped['DSLD ID'].astype(str)

In [61]:
final_df = LS_grouped.merge(PO , on = 'DSLD ID').merge(DSF_grouped , on = 'DSLD ID')
final_df.head()

,DSLD ID,Product Name,Statement,Brand Name,Suggested Use,Ingredient
0,332294,Melatonin 5 mg Strawberry,Great value trusted brand\nSleep #1 drug-free ...,Natrol,Suggested use: Adults; take 1 gummy 30 minutes...,"Calories, Total Carbohydrates, Total Sugars, A..."
1,332295,Melatonin 5 mg Strawberry,Great value trusted brand\nSleep #1 drug-free ...,Natrol,Suggested use: Adults; take 1 gummy 30 minutes...,"Calories, Total Carbohydrates, Total Sugars, A..."
2,332296,Melatonin 5 mg Strawberry,Great value trusted brand\nSleep #1 drug-free ...,Natrol,Suggested use: Adults; take 1 gummy 30 minutes...,"Calories, Total Carbohydrates, Total Sugars, A..."
3,332297,Melatonin 5 mg Strawberry,Sleep\nFall asleep faster\nStay asleep longer\...,Natrol,Suggested Use: Adults; take 1 gummy 30 minutes...,"Calories, Total Carbohydrates, Total Sugars, A..."
4,332298,MelatoninMax 10 mg Blueberry,Sleep\nMaximum strength\nGet better sleep\nWak...,Natrol,Suggested Use: Adults; take 1 gummy 30 minute...,"Calories, Total Carbohydrates, Total Sugars, A..."


In [53]:
final_df.shape

(4780, 6)

In [62]:
final_df['text'] = final_df.drop(columns=['DSLD ID']) \
                           .apply(lambda x: ' '.join(x.astype(str)), axis=1)

In [68]:
final_df[['DSLD ID', 'text']]

,DSLD ID,text
0,332294,Melatonin 5 mg Strawberry Great value trusted ...
1,332295,Melatonin 5 mg Strawberry Great value trusted ...
2,332296,Melatonin 5 mg Strawberry Great value trusted ...
3,332297,Melatonin 5 mg Strawberry Sleep\nFall asleep f...
4,332298,MelatoninMax 10 mg Blueberry Sleep\nMaximum st...
...,...,...
4775,341410,CranRx Liquid Cranberry Fast absorbing liquid ...
4776,341428,Blood Balance Suggested use: As a dietary supp...
4777,341498,Muscle 5 french Vanilla Flavour 24 g Whey prot...
4778,341509,"Congaplex Immune health, Dietary Supplement Th..."


In [69]:
final_df.to_csv('../data/processed_products.csv', index=False, encoding='utf-8')